# Joint MCMC Analysis — DIB 15272 / 15672, C2v / Cs
Loads any backend produced by `run_emcee_asym_both.py`, infers all settings
from the filename, redoes the full convergence analysis, makes the corner plot,
and draws 10 posterior-sample profile overlays.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.gridspec as gridspec
import corner, h5py, os, glob, re, io, subprocess, tempfile
import emcee
from scipy.stats import gaussian_kde, ks_2samp
from scipy.ndimage import gaussian_filter
from scipy.interpolate import interp1d
from scipy.signal import fftconvolve
from IPython.display import display, Image
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display, HTML
display(HTML("<style>.container { width:90% !important; }</style>"))

import sys
sys.path.insert(0, os.path.expanduser('~/DIB'))
from MCMC_convergence_tests import detect_multimodal, ks_stability_test, compute_rhat, format_diagnostics

# ── Matplotlib style ──────────────────────────────────────────────────────────
mpl.rcParams.update(mpl.rcParamsDefault)
plt.rcParams['figure.facecolor'] = 'white'
plt.rc('text', usetex=False)
plt.rc('font', family='serif', size=12)
plt.rc('axes', linewidth=1.5)
plt.rc('xtick', labelsize=11, direction='in', top=True)
plt.rc('ytick', labelsize=11, direction='in', right=True)
plt.rc('xtick.minor', visible=True)
plt.rc('ytick.minor', visible=True)
plt.rc('xtick.major', size=6, pad=4)
plt.rc('xtick.minor', size=3)
plt.rc('ytick.major', size=6)
plt.rc('ytick.minor', size=3)
plt.rc('legend', fontsize=10)
import datetime
import pandas as pd
from scipy.stats import chi2 as chi2_dist


Visualize in first row the direct profile, and in the second row the dP/dx_1 profile, with different error prescriptions

In [29]:
# ════════════════════════════════════════════════════════════════════════════
#  FILE SELECTION
#   Set file_override = '/full/path/to/backend.h5' to pin a specific file.
#   Otherwise the script lists all matching files and picks file_index.
# ════════════════════════════════════════════════════════════════════════════
dib_bands       = ['15672', '15672']   # '15272' or '15672'
which_errs = ['old', 'plate', 'default', 'pca']
fudge  = 1
BalanceErrs = False

for dib_band in dib_bands:
    
    # ── Figure ───────────────────────────────────────────────────────────
    fig, axes = plt.subplots(2, 4, figsize=(19, 10), constrained_layout=True)
            
    for err_model in which_errs:
        if not pgo_available or not models_dib:
            print('No model spectra available — skipping PCA comparison figure.')
        else:
            if dib_band == '15672':
                PCA_FILE     = os.path.expanduser('~/DIB/all_errs/pca_version_15672_narrow.txt')
                OLD_ERR_FILE = os.path.expanduser('~/DIB/all_errs/jackknife_plates_dib_15672.h5')
            if dib_band == '15272':
                PCA_FILE     = os.path.expanduser('~/DIB/all_errs/pca_version_15272_narrow.txt')
                OLD_ERR_FILE = os.path.expanduser('~/DIB/all_errs/jackknife_plates_dib_15272.h5')
            try:
                pca_meas = pd.read_csv(PCA_FILE, sep=r'\s+',
                                       names=['wavelength', 'PC1_1', 'PC1_2', 'PC2_1', 'PC2_2'])
                wav_pca  = pca_meas['wavelength'].values
                with h5py.File(OLD_ERR_FILE, 'r') as ef:
                    dat_pca_dib = pca_meas['PC1_1'].values
                    dat_pca_dT  = pca_meas['PC2_2'].values
                    err_pca_dib = np.sqrt(ef['var'][:, 0]) * fudge
                    err_pca_dT  = np.sqrt(ef['var'][:, 1]) * fudge
                pca_loaded = True
            except Exception as ex:
                print(f'Could not load PCA data: {ex}')
                pca_loaded = False

            if pca_loaded:
                if dib_band == '15672':
                    c_pca          = 20
                if dib_band == '15272':
                    c_pca          = 10
                wav_pca_crop   = wav_pca[c_pca:-c_pca]
                pca_dT_crop    = dat_pca_dT[c_pca:-c_pca]
                err_pca_dT_c   = err_pca_dT[c_pca:-c_pca]
                err_pca_dib_c  = err_pca_dib[c_pca:-c_pca]

                c        = c_crop
                wav_crop = data_wavelength[c:-c]
                wav_off  = wav_crop - (1e8 / CENTRAL_INVCM_BASE)

                # Scalar DoF
                if inferred_sym == 'C2v':
                    n_spec_sc, n_dT_sc = 2, 3
                elif n_blobs == 7:
                    n_spec_sc, n_dT_sc = 3, 4
                else:
                    n_spec_sc, n_dT_sc = 3, 3

                dat_dib_c = data_flux[c:-c]
                dat_dT_c  = data_flux_dT[c:-c]
                err_dib_c = noise_std[c:-c]
                err_dT_c  = noise_std_dT[c:-c]
                med_dib   = np.median(np.array(models_dib), axis=0)
                med_dT    = np.median(np.array(models_dT),  axis=0)

                DARK_RED   = '#8B0000'
                SAMP_COL   = '#4C72B0'
                MED_COL    = '#C44E52'
                PCA_COL    = '#2CA02C'

                # Col 0: DIB profile
                ax = axes19[0]
                for k, m in enumerate(models_dib):
                    ax.plot(wav_crop, m, color=SAMP_COL, alpha=0.15, lw=0.7,
                            label=f'{len(models_dib)} draws' if k == 0 else None)
                ax.plot(wav_crop, med_dib, color=MED_COL, lw=1.8, ls='--', label='Median')
                ax.errorbar(wav_crop, dat_dib_c, yerr=err_dib_c,
                            fmt='o', color=DARK_RED, ecolor=DARK_RED,
                            markersize=2, elinewidth=0.8, capsize=2, alpha=0.8, label='Data')
                cv_s, cr_s, pv_s, dof_s = _cs(dat_dib_c, med_dib, err_dib_c, n_spec_sc)
                ax.annotate(f'$\\chi^2_\\nu = {cr_s:.2f}$, $p = {pv_s:.3f}$',
                            xy=(0.02, 0.97), xycoords='axes fraction', va='top', fontsize=9,
                            bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.8))
                ax.set_title('DIB profile (direct)', fontsize=12)
                ax.set_xlabel(r'$\lambda$ [Å]', fontsize=12)
                ax.set_ylabel('Flux (normalised)', fontsize=12)
                ax.legend(fontsize=8)
                ax.tick_params(which='both', direction='in', top=True, right=True)

                # Col 1: Direct dT (fit data)
                ax = axes19[1]
                for k, m in enumerate(models_dT):
                    ax.plot(wav_crop, m, color=SAMP_COL, alpha=0.15, lw=0.7,
                            label=f'{len(models_dT)} draws' if k == 0 else None)
                ax.plot(wav_crop, med_dT, color=MED_COL, lw=1.8, ls='--', label='Median')
                ax.errorbar(wav_crop, dat_dT_c, yerr=err_dT_c,
                            fmt='o', color=DARK_RED, ecolor=DARK_RED,
                            markersize=2, elinewidth=0.8, capsize=2, alpha=0.8, label='Data (direct)')
                ax.annotate(f'$\\chi^2_\\nu = {cr_dir:.2f}$, $p = {pv_dir:.3f}$',
                            xy=(0.02, 0.97), xycoords='axes fraction', va='top', fontsize=9,
                            bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.8))
                ax.set_title(r'Direct $\partial\mathrm{DIB}/\partial T$  [fit data]', fontsize=12)
                ax.set_xlabel(r'$\lambda$ [Å]', fontsize=12)
                ax.set_ylabel(r'$\Delta$Flux/$\Delta x_1$', fontsize=12)
                ax.legend(fontsize=8)
                ax.tick_params(which='both', direction='in', top=True, right=True)

                # Col 2: PCA dT — same molecular params, re-fitted scalars
                ax = axes19[2]
                for k, m in enumerate(models_pca_dT_list):
                    ax.plot(wav_pca_crop, m, color=PCA_COL, alpha=0.15, lw=0.7,
                            label=f'{len(models_pca_dT_list)} draws (PCA, re-fit)' if k == 0 else None)
                ax.plot(wav_pca_crop, med_pca_dT, color=PCA_COL, lw=1.8, ls='--',
                        label='Median (PCA)')
                ax.errorbar(wav_pca_crop, pca_dT_crop, yerr=err_pca_dT_c,
                            fmt='o', color=DARK_RED, ecolor=DARK_RED,
                            markersize=2, elinewidth=0.8, capsize=2, alpha=0.8, label='PCA data')
                ax.annotate(f'$\\chi^2_\\nu = {cr_pca:.2f}$, $p = {pv_pca:.3f}$',
                            xy=(0.02, 0.97), xycoords='axes fraction', va='top', fontsize=9,
                            bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.8))
                ax.set_title(r'PCA $\partial\mathrm{DIB}/\partial T$  [same mol. params, re-fitted constants]', fontsize=12)
                ax.set_xlabel(r'$\lambda$ [Å]', fontsize=12)
                ax.set_ylabel(r'$\Delta$Flux/$\Delta x_1$ (PCA)', fontsize=12)
                ax.legend(fontsize=8)
                ax.tick_params(which='both', direction='in', top=True, right=True)

                fig19.suptitle(
                    f'DIB {inferred_dib}  \u00b7  {inferred_sym}  \u00b7  '
                    'PCA temperature derivative — same molecular parameters, re-fitted linear scalars',
                    fontsize=11
                )
                if save_fig:
                    fig19.savefig(os.path.join(FIG_DIR, f'profiles_pca_comparison.pdf'),
                                  bbox_inches='tight')
                plt.show()


Found 11 matching file(s) — sorted newest first:
  [0]  15672_run_DIB15672_SymmetryC2v_BCTrue_F1p0_DTrue_FlatFalse_SpecTrue_dTTrue_covFalse_nonlinFalse_tauSlope0p15_alphaSlope0p15_errDefault_trunc10_balanceErrTrue_dTcenterNone_centeroffset0p15_test.h5
  [1]  15672_run_DIB15672_SymmetryC2v_BCTrue_F1p0_DTrue_FlatFalse_SpecTrue_dTTrue_covFalse_nonlinFalse_tauSlope0p15_alphaSlope0p15_errPlate_trunc10_balanceErrTrue_dTcenterNone_centeroffset0p15_test.h5  <-- selected
  [2]  15672_run_SymmetryC2v_BCTrue_F1p0_DTrue_FlatFalse_SpecTrue_dTTrue_covFalse_nonlinFalse_tauSlope0p15_alphaSlope0p02_errPlate_trunc10_balanceErrFalse_dTcenterTrue_new_priors.h5
  [3]  15672_run_SymmetryC2v_BCTrue_F1p0_DTrue_FlatFalse_SpecTrue_dTTrue_covFalse_nonlinFalse_tauSlope0p15_alphaSlope0p02_errPlate_trunc10_balanceErrFalse_dTcenterFalse_new_priors.h5
  [4]  15672_run_SymmetryC2v_BCTrue_F1p0_DTrue_FlatFalse_SpecTrue_dTTrue_covFalse_nonlinFalse_tauSlope0p15_alphaSlope0p02_errPlate_trunc10_balanceErrTrue_dTcenterFalse_